# RAY-IMAGE — N1 VAE Diagnostic (Current Step)

This notebook is intentionally limited to the current experiment. It trains the VAE, runs N1 reconstruction/latent diagnostics, shows the reconstruction images, and then stops. Do **not** run generator training yet.

**Current mission:** verify that the VAE preserves the toy task's color and shape information before the FIRST_IMAGE generator experiment.

Select **Runtime → Change runtime type → T4 GPU** first.

In [ ]:
%cd /content
!rm -rf anime-ai-companion
!git clone https://github.com/Rishidev-20thcenturey/anime-ai-companion.git
%cd /content/anime-ai-companion
!git fetch origin arena/01a07cdc-anime-ai-companion
!git checkout arena/01a07cdc-anime-ai-companion
!pip install -q -r requirements.txt
print('Checked out active RAY-IMAGE branch.')

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU attached. Select a GPU runtime and reconnect.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
!python -m ray_image.train_smoke

In [ ]:
!rm -rf data/toy
!python tools/make_toy_dataset.py --output data/toy --samples 2048 --size 64 --seed 1337
print('Toy dataset ready.')

In [ ]:
!python -m ray_image.train_vae \
  --manifest data/toy/manifest.jsonl \
  --steps 1200 \
  --batch-size 32 \
  --save /content/ray_vae_v0_2.pt \
  --seed 0

In [ ]:
# N1 diagnostic — no generator training
!rm -rf /content/n1
!mkdir -p /content/n1
!python -m ray_image.probe_vae_latents \
  --vae-checkpoint /content/ray_vae_v0_2.pt \
  --manifest data/toy/manifest.jsonl \
  --outdir /content/n1 \
  --size 64 \
  --seed 1337 \
  --stats-samples 512 \
  --stats-batch 32
print('\nReconstruction files:')
!ls -1 /content/n1/reconstructions

In [ ]:
from pathlib import Path
from PIL import Image, ImageDraw
from IPython.display import display
files = sorted(Path('/content/n1/reconstructions').glob('*.png'))
thumbs = [Image.open(p).convert('RGB').resize((192, 192)) for p in files]
cols = 4
rows = (len(thumbs) + cols - 1) // cols
sheet = Image.new('RGB', (cols * 192, rows * 220), 'white')
draw = ImageDraw.Draw(sheet)
for i, (p, im) in enumerate(zip(files, thumbs)):
    x = (i % cols) * 192
    y = (i // cols) * 220
    sheet.paste(im, (x, y))
    draw.text((x + 5, y + 196), p.stem, fill='black')
display(sheet)
print('\nN1 stats file:')
print(Path('/content/n1/vae_latent_stats.json').read_text())
print('\nSTOP HERE. Send the N1 metrics, latent stats, and reconstruction image grid back to ChatGPT.')